# CUDA Hello World

**Laboratorio de aceleración de cómputo · Kaggle GPU**

Esta notebook documenta la primera ejecución de un kernel CUDA en una GPU NVIDIA. La salida de las celdas sirve como evidencia de trazabilidad en Kaggle; la versión mantenible del programa está en `hello_cuda.cu` dentro del repositorio.

## Objetivos

- Confirmar que Kaggle asignó una GPU NVIDIA.
- Comprobar que el compilador `nvcc` está disponible.
- Escribir, compilar y ejecutar un kernel CUDA mínimo.
- Observar la identificación del bloque y del hilo que ejecutan el kernel.

> **Antes de ejecutar:** activa el acelerador **GPU** en la configuración de la notebook.

In [ ]:
!nvidia-smi
!nvcc --version

## 2. Kernel Hello World

Se lanza un bloque con un único hilo. El kernel escribe un mensaje desde el device y la sincronización permite esperar su finalización antes de cerrar el programa.

In [ ]:
%%writefile hello_cuda.cu
#include <cuda_runtime.h>
#include <cstdio>

#define CUDA_CHECK(call) do { \
    cudaError_t error = (call); \
    if (error != cudaSuccess) { \
        std::fprintf(stderr, "CUDA error: %s\n", cudaGetErrorString(error)); \
        return 1; \
    } \
} while (false)

__global__ void helloKernel() {
    printf("Hello World desde CUDA | bloque %d | hilo %d\n", blockIdx.x, threadIdx.x);
}

int main() {
    cudaDeviceProp device{};
    CUDA_CHECK(cudaGetDeviceProperties(&device, 0));
    printf("GPU activa: %s\n", device.name);
    helloKernel<<<1, 1>>>();
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());
    printf("Ejecucion finalizada correctamente.\n");
    return 0;
}

In [ ]:
!nvcc -O2 hello_cuda.cu -o hello_cuda
!./hello_cuda

## Resultado esperado

La salida debe mostrar el nombre de la GPU, un mensaje desde el hilo `(bloque 0, hilo 0)` y la confirmación de finalización. Conserva la salida de esta celda al guardar una versión de la notebook en Kaggle.